# ABT 1 Creation: Imputation of `TotalCharges`

Decision to run a Linear Regression to help qith the missing values on this variable.

In [0]:
%pip install feature_engine
%pip install xgboost lightgbm catboost

In [0]:
# Tables
tb_fs = 'feature_store_train'
tb_ts = 'target_store_train'

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [0]:
from sklearn import model_selection

df_fs = spark.table(f'workspace.default.{tb_fs}').toPandas()
df_ts = spark.table(f'workspace.default.{tb_ts}').toPandas()

df = df_fs.merge(df_ts, on='customerID')

# X = df.drop(['id', 'target'], axis=1)
# y = df['target']

df.head()


In [0]:
%sql
DROP TABLE IF EXISTS workspace.default.abt_1;
CREATE TABLE IF NOT EXISTS workspace.default.abt_1 AS
SELECT
    t1.*,
    t2.TargetChurn,
    RAND() AS random_pct_decimal
FROM workspace.default.feature_store_train AS t1
LEFT JOIN workspace.default.target_store_train AS t2
    ON t1.customerID = t2.customerID ;


In [0]:
df_clean = df.dropna()

df_clean.describe().T

## Analysis (move to a different notebook)

In [0]:
target_col = 'TargetChurn' #'TotalCharges'
drop_cols = ['customerID', 'TotalSpend','TotalSpendPct', 'ContractTier', 'tenureGroup','ServiceTier', 'CustomerFeedback_adj']
feature_df = df_clean.drop(columns=drop_cols + [target_col])

is_binary = feature_df.apply(lambda s: s.dropna().isin([0, 1]).all())
binary_cols  = is_binary[is_binary].index.tolist()
numeric_cols = feature_df.select_dtypes(include=[np.number]).columns.tolist()
cont_cols    = [c for c in numeric_cols if c not in binary_cols]

print(f'binary: {len(binary_cols)}  continuous: {len(cont_cols)}')
print('binary  :', binary_cols[:20], '...' if len(binary_cols) > 20 else '')
print('cont.   :', cont_cols[:20],  '...' if len(cont_cols) > 20 else '')

In [0]:
# Drop constant columns, they break corr() and add nothing
usable = [c for c in feature_df.columns if feature_df[c].nunique(dropna=True) > 1]
dropped_constant = sorted(set(feature_df.columns) - set(usable))
print("dropped constant:", dropped_constant)

num_df = feature_df[usable].apply(pd.to_numeric, errors="coerce")

In [0]:
corr = num_df.corr(method="pearson")

plt.figure(figsize=(18, 14))
sns.heatmap(
    corr,
    cmap="coolwarm", center=0, vmin=-1, vmax=1,
    square=True,
    linewidths=0.3, linecolor="white",
    cbar_kws={"shrink": 0.7},
)
plt.title("Correlation matrix: feature_store_train_clean (binary + continuous)")
plt.tight_layout()
plt.show()

In [0]:
new_Target = 'TotalCharges'
target_corr = (
    corr[new_Target]
    .drop(new_Target)
    .sort_values(key=lambda s: s.abs(), ascending=False)
)
# target_corr.to_string()
target_corr

In [0]:
MIN_CORR = 0.0

candidates = target_corr[target_corr.abs() >= MIN_CORR]
print(f"{len(candidates)} candidates above |corr| >= {MIN_CORR}")
print(candidates.to_string())

### VIF (best for linear models)

VIF directly measures "how much is this feature explained by all the others". 

VIF > 5 = worth reviewing, VIF > 10 = drop.

In [0]:
# from scipy.linalg import qr

# X = num_df[candidates.index].fillna(0).astype(float).values

# # Pivoted QR — reveals which columns are dependent on earlier ones
# Q, R, piv = qr(X, mode="economic", pivoting=True)
# diag = np.abs(np.diag(R))
# tol = diag.max() * 1e-10
# rank = (diag > tol).sum()
# dependent = [candidates.index[piv[i]] for i in range(rank, len(piv))]

# print(f"matrix rank   : {rank}")
# print(f"n features    : {len(candidates)}")
# print(f"dependent cols: {dependent}")

In [0]:
# drop_for_dependency = dependent + ['isPaymentMethodCheck', 'hasSecurityBundle', 'nbrInternetServicesAddOnQtyPct', 'isM2M_internetOnly']

# clean_candidates = [c for c in candidates.index if c not in drop_for_dependency]
# print(f"{len(clean_candidates)} features left")

# print("n clean candidates:", len(clean_candidates))
# print(clean_candidates)

In [0]:
# from statsmodels.stats.outliers_influence import variance_inflation_factor
# from statsmodels.tools.tools import add_constant

# # Use the clean feature matrix (not the corr matrix)
# X = num_df[clean_candidates].fillna(0).astype(float)
# X_const = add_constant(X, has_constant="add")

# vif = pd.DataFrame({
#     "feature": X_const.columns,
#     "VIF": [variance_inflation_factor(X_const.values, i)
#             for i in range(X_const.shape[1])],
# })
# vif = vif[vif["feature"] != "const"].sort_values("VIF", ascending=False)
# print(vif.to_string(index=False))

In [0]:
# vif_pruned = vif[vif["VIF"] < 10]
# print(f"{len(vif_pruned)} features left after pruning VIF > 10")
# print(vif_pruned["feature"].to_list())
# # Drop columns with high VIF
# drop_for_vif = [c for c in clean_candidates if c not in vif_pruned["feature"].to_list()]
# print(f"dropped {len(drop_for_vif)} columns for high VIF")
# print(drop_for_vif)

# vif_pruned

## Build the ML Model

In [0]:
REMOVE_VARS = [
    'TotalSpend',       # = tenure × MonthlyCharges
    'TotalSpendPct',    # = TotalCharges / TotalSpend
    'tenure_calc',      # = TotalCharges / MonthlyCharges
    'chargeRatio',      # ratio with MonthlyCharges — borderline, safe to keep but consider
    'numServices',      # not leaky but redundant with add-ons
    'chargesPerService',# = MonthlyCharges / numServices
    'nbrInternetServicesAddOnQtyPct',
    'tenure_calc',
]

KEEP_VARS = [
    'nbrTenure',
    'MonthlyCharges',
    'isContractTwoYear',
    'isContractOneYear',
    'TotalSpend',
]

# Recompute correlation against TotalCharges (not TargetChurn)
TARGET_REG = 'TotalCharges'
# num_df_reg = num_df.drop(columns=[c for c in REMOVE_VARS if c in num_df.columns], errors='ignore').copy()
num_df_reg = num_df[[c for c in KEEP_VARS if c in num_df.columns]].copy()

target_corr_reg = (
    num_df_reg.assign(**{TARGET_REG: df_clean[TARGET_REG]})
              .corr(method='pearson')[TARGET_REG]
              .drop(TARGET_REG)
              .sort_values(key=abs, ascending=False)
)
print(target_corr_reg.head(20).to_string())

In [0]:
MIN_CORR = 0.0

feature_list_reg = target_corr_reg[target_corr_reg.abs() >= MIN_CORR].index.tolist()
print(f"{len(feature_list_reg)} candidates above |corr| >= {MIN_CORR}")

X = num_df_reg[feature_list_reg].fillna(0).astype(float)
y = df_clean[TARGET_REG].astype(float)

# Align indices (both come from df_clean, so this should already work)
print("X:", X.shape, "y:", y.shape)

In [0]:

import mlflow
import mlflow.sklearn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
import numpy as np
import logging


# Suppress the noisy MLflow tracking context registry warnings
logging.getLogger("mlflow.tracking.context.registry").setLevel(logging.ERROR)


# Set the registry to Unity Catalog
mlflow.set_experiment(experiment_id=1316272881415465)

# Your feature list for TotalCharges prediction
# (Make sure 'TotalSpend', 'TotalSpendPct', 'tenure_calc' are NOT in this list)
# feature_list_reg = ...

X = num_df_reg[feature_list_reg].fillna(0).astype(float)
y = df_clean["TotalCharges"].astype(float)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [0]:
# with mlflow.start_run(run_name="total_charges_baseline_ols") as run:
#     model = LinearRegression()
#     model.fit(X_train, y_train)

#     train_preds = model.predict(X_train)
#     test_preds  = model.predict(X_test)

#     train_rmse = np.sqrt(mean_squared_error(y_train, train_preds))
#     train_mae  = mean_absolute_error(y_train, train_preds)
#     train_r2   = r2_score(y_train, train_preds)
#     train_mape = mean_absolute_percentage_error(y_train, train_preds)

#     test_rmse  = np.sqrt(mean_squared_error(y_test, test_preds))
#     test_mae   = mean_absolute_error(y_test, test_preds)
#     test_r2    = r2_score(y_test, test_preds)
#     test_mape = mean_absolute_percentage_error(y_test, test_preds)

#     mlflow.log_param("model_type", "LinearRegression")
#     mlflow.log_param("n_features", len(feature_list_reg))

#     mlflow.log_metrics({
#         "train_rmse": train_rmse, "train_mae": train_mae, "train_r2": train_r2, "train_mape": train_mape,
#         "test_rmse":  test_rmse,  "test_mae":  test_mae,  "test_r2":  test_r2, "test_mape":  test_mape,
#     })

#     mlflow.sklearn.log_model(
#         sk_model=model,
#         name="model",
#         registered_model_name="workspace.default.total_charges_baseline_ols",
#         input_example=X_train.iloc[[0]],
#     )

#     print(f"OLS Train || RMSE: {train_rmse:.2f}, MAE: {train_mae:.2f}, R²: {train_r2:.4f}, MAPE: {train_mape:.4f}")
#     print(f"OLS  Test || RMSE: {test_rmse:.2f}, MAE: {test_mae:.2f}, R²: {test_r2:.4f}, MAPE: {test_mape:.4f}")

## Ridge: The Key Parameters That Control Ridge

### Parameter 1: Regularization Strength (α or regParam)
- This is the single most important parameter in Ridge:
    - In Scikit-Learn: alpha (e.g. Ridge(alpha=1.0))
    - In PySpark MLlib: regParam (e.g. LinearRegression(regParam=0.1))

- When alpha = 0: The penalty is turned off --> Identical to standard OLS Linear Regression
- Small alpha( e.g. 0.01-1.0): Slight shrinkage --> Calms down correlation, stabilizes weights, preserves high accuracy
- Large alpha (e.g. 100+): Heavy shrinkage --> Coefficients shrink close to 0; model becomes too simple (underfitting)

> In our dataset, because the relationship is physically near-deterministic (tenure * MonthlyCharges), a small α (like 0.1 or 1.0) gives you all the stability of Ridge without sacrificing any 𝑅².

### Parameter 2: Feature Standardization (standardize or StandardScaler)
- Why this matters for Ridge (unlike OLS):
    - In OLS, feature scale doesn't change the predictions.
    - In Ridge, because we penalize 𝛽², a feature with large numerical values (like TotalCharges in thousands or tenure * MonthlyCharges up to 8,000) would naturally get smaller 𝛽 0/1 binary flag!
    - Therefore, features must be on the same scale before calculating the penalty, so all features are penalized fairly.

- In PySpark MLlib: Handled automatically by setting standardization=True (default in Spark LinearRegression).
- In Scikit-Learn: Handled by passing data through StandardScaler().

### Parameter 3: The Mixing Parameter (elasticNetParam — PySpark specific)

In Databricks / Spark MLlib, the LinearRegression class handles OLS, Ridge, and Lasso under one roof using elasticNetParam:
- elasticNetParam = 0.0 → Pure Ridge (L2)
- elasticNetParam = 1.0 → Pure Lasso (L1)
- elasticNetParam = 0.5 → ElasticNet (a blend of both)

In [0]:

# parameter_alpha = 0.1

# with mlflow.start_run(run_name="total_charges_ridge") as run:
#     # Train
#     model = Ridge(alpha=parameter_alpha)
#     model.fit(X_train, y_train)

#     # Predict
#     train_preds = model.predict(X_train)
#     test_preds = model.predict(X_test)

#     # 3. Calculate Training Metrics
#     train_rmse = np.sqrt(mean_squared_error(y_train, train_preds))
#     train_mae = mean_absolute_error(y_train, train_preds)
#     train_r2 = r2_score(y_train, train_preds)
#     train_mape = mean_absolute_percentage_error(y_train, train_preds)

#     # 4. Calculate Testing Metrics
#     test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))
#     test_mae = mean_absolute_error(y_test, test_preds)
#     test_r2 = r2_score(y_test, test_preds)
#     test_mape = mean_absolute_percentage_error(y_test, test_preds)

#     # Log params and metrics
#     mlflow.log_param("alpha", parameter_alpha)
#     mlflow.log_param("n_features", len(feature_list_reg))

#     # Log test metrics batch at once using a dictionary
#     mlflow.log_metrics({
#         "train_rmse": train_rmse,
#         "train_mae": train_mae,
#         "train_r2": train_r2,
#         "train_mape": train_mape,
#         "test_rmse": test_rmse,
#         "test_mae": test_mae,
#         "test_r2": test_r2,
#         "test_mape":  test_mape,
#     })

#     # Log the model to Unity Catalog
#     # Replace 'main.default' with your actual catalog and schema if different
#     mlflow.sklearn.log_model(
#         sk_model=model,
#         # artifact_path="model", # <-- Deprecated
#         name="model",
#         registered_model_name="workspace.default.total_charges_ridge_model",
#         input_example=X_train.iloc[[0]]
#     )

#     print(f"Ridge Train || RMSE: {train_rmse:.2f}, MAE: {train_mae:.2f}, R²: {train_r2:.4f}, MAPE: {train_mape:.4f}")
#     print(f"Ridge Test ||RMSE: {test_rmse:.2f}, MAE: {test_mae:.2f}, R²: {test_r2:.4f}, MAPE: {test_mape:.4f}")

In [0]:
# REUSABLE MODEL TEMPLATE
def train_and_log(
    model_name,
    estimator,
    param_grid=None,          # None -> plain fit; dict/list -> GridSearchCV
    cv=5,
    scoring='neg_mean_squared_error',
    X_train=None, y_train=None, X_test=None, y_test=None,
    max_tuning_runs=20,
):
    # Autolog: logs params, metrics, model signature, and GridSearchCV child runs
    mlflow.sklearn.autolog(exclusive=False, max_tuning_runs=max_tuning_runs)

    with mlflow.start_run(run_name=model_name) as run:
        # ---- Train ----
        if param_grid is not None:
            model = GridSearchCV(
                estimator=estimator,
                param_grid=param_grid,
                cv=cv,
                scoring=scoring,
                n_jobs=-1,
            )
            model.fit(X_train, y_train)
            best_model = model.best_estimator_
            mlflow.log_params(model.best_params_)
            mlflow.log_metric('best_cv_score', model.best_score_)
            print(f'Best params: {model.best_params_}')
        else:
            model = estimator
            model.fit(X_train, y_train)
            best_model = model

        # ---- Predict + metrics (using the best estimator) ----
        train_preds = best_model.predict(X_train)
        test_preds = best_model.predict(X_test)

        mlflow.log_metrics({
            'train_rmse': np.sqrt(mean_squared_error(y_train, train_preds)),
            'train_mae': mean_absolute_error(y_train, train_preds),
            'train_r2': r2_score(y_train, train_preds),
            'train_mape': mean_absolute_percentage_error(y_train, train_preds),
            'test_rmse': np.sqrt(mean_squared_error(y_test, test_preds)),
            'test_mae': mean_absolute_error(y_test, test_preds),
            'test_r2': r2_score(y_test, test_preds),
            'test_mape': mean_absolute_percentage_error(y_test, test_preds),
        })

        # ---- Register best model ----
        mlflow.sklearn.log_model(
            sk_model=best_model,
            name='model',
            registered_model_name=f'workspace.default.{model_name}',
            input_example=X_train.iloc[[0]],
        )

        print(f'==> Model: {model_name}')
        print(f"Train || RMSE: {train_rmse:.2f}, MAE: {train_mae:.2f}, R²: {train_r2:.4f}, MAPE: {train_mape:.4f}")
        print(f"Test ||RMSE: {test_rmse:.2f}, MAE: {test_mae:.2f}, R²: {test_r2:.4f}, MAPE: {test_mape:.4f}")

In [0]:
# Plain Ridge (no tuning)
train_and_log(
    model_name='total_charges_ridge',
    estimator=Ridge(alpha=0.1),
    X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test,
)

# Ridge with GridSearchCV
train_and_log(
    model_name='total_charges_ridge_tuned',
    estimator=Ridge(),
    param_grid={'alpha': [0.01, 0.1, 1.0, 10.0, 100.0]},
    X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test,
)

# OLS baseline
train_and_log(
    model_name='total_charges_baseline_ols',
    estimator=LinearRegression(),
    X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test,
)

# Model Comparison

In [0]:

id = 1316272881415465
max_results = 1000  # Maximum number of rows to retrieve
runs = mlflow.search_runs(experiment_ids=[id], max_results=max_results)

# Display dataframe in cell output
display(runs)


# Feature Importance

In [0]:
import mlflow

# Import Model
# model = mlflow.sklearn.load_model("models:/workspace.default.total_charges_baseline_ols/1")
model = mlflow.sklearn.load_model("models:/workspace.default.total_charges_ridge_model/8")

model

In [0]:
# 2. Extract the regressor and feature names
# Case A: If it's a Pipeline (e.g. Pipeline([('scaler', StandardScaler()), ('regressor', LinearRegression())]))
if hasattr(model, "named_steps"):
    # Grab the last step (the regressor)
    regressor = model.named_steps[list(model.named_steps.keys())[-1]]
    # Get feature names from the pipeline if available, or pass your X_train.columns
    if hasattr(model, "feature_names_in_"):
        feature_names = model.feature_names_in_
    else:
        feature_names = X_train.columns  # fallback to your training dataframe columns
# Case B: If it's a standalone model (LinearRegression or Ridge directly)
else:
    regressor = model
    feature_names = getattr(model, "feature_names_in_", X_train.columns)

# 3. Create a clean DataFrame of Coefficients
df_importance = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': regressor.coef_,
    'Abs_Importance': abs(regressor.coef_)
}).sort_values(by='Abs_Importance', ascending=False).reset_index(drop=True)

# 4. Display the table in Databricks
display(df_importance)

# 5. Plot the top features
plt.figure(figsize=(10, 8))
df_importance.head(15).sort_values(by='Abs_Importance', ascending=True).plot(
    x='Feature', 
    y='Coefficient', 
    kind='barh', 
    legend=False, 
    color='steelblue',
    edgecolor='black'
)
plt.title("Top Feature Coefficients (Linear Model)", fontsize=14)
plt.xlabel("Coefficient Value")
plt.axvline(0, color='red', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.show()